# Статистический анализ Olist E-commerce

В этом ноутбуке проводится статистическая проверка гипотез, сформулированных по результатам исследовательского анализа данных Olist E-commerce.

Цель анализа — определить, являются ли обнаруженные в EDA взаимосвязи статистически значимыми, оценить направление и силу обнаруженных связей и сформировать формальный статистический вывод.

Проверяются четыре гипотезы, сформулированные по результатам EDA:

- своевременность доставки и оценка заказа;
- расстояние между продавцом и покупателем и стоимость доставки;
- расстояние до наиболее удалённого продавца и время доставки;
- количество installments и стоимость товаров в заказе.

Наблюдаемые зависимости рассматриваются как статистические ассоциации и сами по себе не интерпретируются как причинно-следственные связи.

## 1. Настройка окружения

In [1]:
import numpy as np
import pandas as pd

from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

from sqlalchemy import create_engine, text

In [2]:
ALPHA = 0.05
CONFIDENCE_LEVEL = 0.95

In [3]:
engine = create_engine(
    "postgresql+psycopg2:///olist_analytics"
)

In [4]:
def query(sql):
    return pd.read_sql(
        text(sql),
        engine,
    )

## 2. Подготовка данных для статистического анализа

Для каждой статистической гипотезы формируется отдельная аналитическая выборка.

Это позволяет явно контролировать единицу анализа, правила фильтрации и набор используемых признаков.

Подготавливаются четыре выборки:

- доставленные заказы с оценкой и отклонением от ожидаемой даты доставки;
- пары `order_id × seller_id` с расстоянием и стоимостью доставки;
- доставленные заказы с расстоянием до наиболее удалённого продавца и временем доставки;
- доставленные заказы с одной валидной платежной записью по кредитной карте для анализа installments.

In [5]:
delivery_review = query("""
    WITH reviews_agg AS (
        SELECT
            order_id,

            AVG(review_score)::DOUBLE PRECISION
                AS avg_order_review_score

        FROM analytics.reviews

        GROUP BY
            order_id
    )

    SELECT
        o.order_id,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        r.avg_order_review_score

    FROM analytics.orders AS o

    JOIN reviews_agg AS r
        USING (order_id)

    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
""")

In [6]:
delivery_review[
    "order_delivered_customer_date"
] = pd.to_datetime(
    delivery_review[
        "order_delivered_customer_date"
    ]
)

delivery_review[
    "order_estimated_delivery_date"
] = pd.to_datetime(
    delivery_review[
        "order_estimated_delivery_date"
    ]
)

delivery_review[
    "delivery_deviation_days"
] = (
    delivery_review[
        "order_delivered_customer_date"
    ].dt.normalize()
    -
    delivery_review[
        "order_estimated_delivery_date"
    ].dt.normalize()
).dt.days

In [7]:
assert not delivery_review["order_id"].duplicated().any()

In [8]:
delivery_review[
    [
        "delivery_deviation_days",
        "avg_order_review_score",
    ]
].isna().sum()

delivery_deviation_days    0
avg_order_review_score     0
dtype: int64

In [9]:
order_seller_geo = query("""
    WITH order_seller_metrics AS (
        SELECT
            order_id,
            seller_id,

            SUM(freight_value)::DOUBLE PRECISION
                AS pair_freight_value

        FROM analytics.order_items

        GROUP BY
            order_id,
            seller_id
    )

    SELECT
        osm.order_id,
        osm.seller_id,
        osm.pair_freight_value,

        c.customer_state,
        s.seller_state,

        customer_geo.geolocation_state
            AS customer_geo_state,

        seller_geo.geolocation_state
            AS seller_geo_state,

        customer_geo.geolocation_lat
            AS customer_lat,

        customer_geo.geolocation_lng
            AS customer_lng,

        seller_geo.geolocation_lat
            AS seller_lat,

        seller_geo.geolocation_lng
            AS seller_lng

    FROM order_seller_metrics AS osm

    JOIN analytics.orders AS o
        USING (order_id)

    JOIN analytics.customers AS c
        USING (customer_id)

    JOIN analytics.sellers AS s
        USING (seller_id)

    LEFT JOIN analytics.geolocation AS customer_geo
        ON c.customer_zip_code_prefix =
           customer_geo.geolocation_zip_code_prefix

    LEFT JOIN analytics.geolocation AS seller_geo
        ON s.seller_zip_code_prefix =
           seller_geo.geolocation_zip_code_prefix

    WHERE o.order_status = 'delivered'
""")

In [10]:
assert not order_seller_geo.duplicated(
    ["order_id", "seller_id"]
).any()

In [11]:
geo_pairs = (
    order_seller_geo
    .dropna(
        subset=[
            "customer_lat",
            "customer_lng",
            "seller_lat",
            "seller_lng",
            "customer_geo_state",
            "seller_geo_state",
        ]
    )
    .copy()
)

geo_pairs = geo_pairs[
    geo_pairs["customer_state"].eq(
        geo_pairs["customer_geo_state"]
    )
    &
    geo_pairs["seller_state"].eq(
        geo_pairs["seller_geo_state"]
    )
    &
    geo_pairs["customer_lat"].between(-34, 6)
    &
    geo_pairs["customer_lng"].between(-74, -28)
    &
    geo_pairs["seller_lat"].between(-34, 6)
    &
    geo_pairs["seller_lng"].between(-74, -28)
].copy()

In [12]:
customer_lat = np.radians(
    geo_pairs["customer_lat"]
)

customer_lng = np.radians(
    geo_pairs["customer_lng"]
)

seller_lat = np.radians(
    geo_pairs["seller_lat"]
)

seller_lng = np.radians(
    geo_pairs["seller_lng"]
)

cos_angle = (
    np.sin(customer_lat)
    * np.sin(seller_lat)
    +
    np.cos(customer_lat)
    * np.cos(seller_lat)
    * np.cos(
        seller_lng - customer_lng
    )
)

geo_pairs["distance_km"] = (
    6371
    * np.arccos(
        np.clip(
            cos_angle,
            -1,
            1,
        )
    )
)

In [13]:
all_pairs_per_order = (
    order_seller_geo
    .groupby("order_id")
    .size()
    .rename("all_pairs")
)

valid_pairs_per_order = (
    geo_pairs
    .groupby("order_id")
    .size()
    .rename("valid_pairs")
)

geo_order_coverage = (
    pd.concat(
        [
            all_pairs_per_order,
            valid_pairs_per_order,
        ],
        axis=1,
    )
    .fillna(0)
)

complete_geo_order_ids = (
    geo_order_coverage[
        geo_order_coverage["all_pairs"]
        ==
        geo_order_coverage["valid_pairs"]
    ]
    .index
)

In [14]:
order_distance = (
    geo_pairs[
        geo_pairs["order_id"].isin(
            complete_geo_order_ids
        )
    ]
    .groupby(
        "order_id",
        as_index=False,
    )
    .agg(
        max_seller_distance_km=(
            "distance_km",
            "max",
        )
    )
)

In [15]:
delivery_time = query("""
    SELECT
        order_id,
        order_purchase_timestamp,
        order_delivered_customer_date

    FROM analytics.orders

    WHERE order_status = 'delivered'
      AND order_purchase_timestamp IS NOT NULL
      AND order_delivered_customer_date IS NOT NULL
""")

In [16]:
delivery_time[
    "order_purchase_timestamp"
] = pd.to_datetime(
    delivery_time[
        "order_purchase_timestamp"
    ]
)

delivery_time[
    "order_delivered_customer_date"
] = pd.to_datetime(
    delivery_time[
        "order_delivered_customer_date"
    ]
)

delivery_time[
    "delivery_time_days"
] = (
    delivery_time[
        "order_delivered_customer_date"
    ]
    -
    delivery_time[
        "order_purchase_timestamp"
    ]
).dt.total_seconds() / 86400

In [17]:
distance_delivery = (
    order_distance
    .merge(
        delivery_time[
            [
                "order_id",
                "delivery_time_days",
            ]
        ],
        on="order_id",
        how="inner",
        validate="1:1",
    )
)

In [18]:
installment_orders = query("""
    WITH single_payment_orders AS (
        SELECT
            p.order_id

        FROM analytics.payments AS p

        JOIN analytics.orders AS o
            USING (order_id)

        WHERE o.order_status = 'delivered'

        GROUP BY
            p.order_id

        HAVING COUNT(*) = 1
    ),

    order_gmv AS (
        SELECT
            order_id,

            SUM(price)::DOUBLE PRECISION
                AS product_gmv

        FROM analytics.order_items

        GROUP BY
            order_id
    )

    SELECT
        p.order_id,
        p.payment_installments
            AS installments,
        g.product_gmv

    FROM analytics.payments AS p

    JOIN single_payment_orders AS spo
        USING (order_id)

    JOIN order_gmv AS g
        USING (order_id)

    WHERE p.payment_type = 'credit_card'
      AND p.payment_installments > 0
""")

In [19]:
assert not installment_orders[
    "order_id"
].duplicated().any()

assert installment_orders[
    [
        "installments",
        "product_gmv",
    ]
].notna().all().all()

In [20]:
samples_summary = pd.DataFrame({
    "sample": [
        "delivery_review",
        "geo_pairs",
        "distance_delivery",
        "installment_orders",
    ],
    "rows": [
        len(delivery_review),
        len(geo_pairs),
        len(distance_delivery),
        len(installment_orders),
    ],
})

samples_summary

,sample,rows
0,delivery_review,95824
1,geo_pairs,96754
2,distance_delivery,95411
3,installment_orders,71841


Каждая выборка подготовлена в соответствии с единицей анализа конкретной гипотезы. Далее статистические проверки выполняются отдельно для каждой выборки без изменения правил фильтрации, заданных на этапе подготовки данных.

## 3. Методология статистического анализа

Для проверки сформулированных гипотез используется уровень значимости

$$
\alpha = 0.05.
$$

Основной мерой связи между числовыми и порядковыми признаками является коэффициент ранговой корреляции Спирмена (`Spearman ρ`).

Он выбран потому, что исследуемые признаки имеют выраженные асимметричные распределения, содержат экстремальные значения, а обнаруженные в EDA зависимости не обязательно являются линейными.

Для каждой гипотезы будут представлены:

- размер аналитической выборки;
- коэффициент корреляции Спирмена;
- исходный `p-value`;
- направление и величина наблюдаемой связи.

Несмотря на то, что гипотезы сформулированы с ожидаемым направлением связи, для статистических проверок используются двусторонние `p-value`. Направление зависимости оценивается по знаку коэффициента корреляции.

После проверки всех четырёх гипотез `p-value` будут скорректированы методом Холма для контроля семейной вероятности ошибки первого рода при множественных проверках.

Из-за большого размера выборок статистическая значимость сама по себе не рассматривается как достаточное свидетельство практически важной связи. Поэтому основное внимание также уделяется величине коэффициента корреляции.

Гипотезы были сформулированы на основе EDA, выполненного на тех же данных. Поэтому полученные `p-value` следует интерпретировать как формальную статистическую оценку обнаруженных зависимостей, а не как независимое подтверждение на новой выборке.

In [21]:
def spearman_test(
    data,
    x,
    y,
):
    sample = (
        data[
            [
                x,
                y,
            ]
        ]
        .dropna()
    )

    result = stats.spearmanr(
        sample[x],
        sample[y],
        alternative="two-sided",
    )

    return {
        "n": len(sample),
        "rho": result.statistic,
        "p_value": result.pvalue,
    }

In [22]:
test_results = {}

## 4. Гипотеза 1: своевременность доставки и оценка заказа

Проверим статистическую связь между отклонением фактической даты доставки от ожидаемой даты (`delivery_deviation_days`) и средней оценкой заказа (`avg_order_review_score`).

Для `delivery_deviation_days`:

- отрицательное значение — заказ доставлен раньше обещанной даты;
- `0` — заказ доставлен в обещанную дату;
- положительное значение — заказ доставлен позже обещанной даты.

**H₀:** между отклонением от ожидаемой даты доставки и оценкой заказа отсутствует монотонная связь.

**H₁:** между отклонением от ожидаемой даты доставки и оценкой заказа существует монотонная связь.

На основании результатов EDA ожидается отрицательное направление связи: более поздняя доставка относительно обещанного срока соответствует более низкой оценке.

Единица анализа — доставленный заказ.

Для проверки используется двусторонний тест значимости коэффициента ранговой корреляции Спирмена.

In [23]:
delivery_review[
    [
        "delivery_deviation_days",
        "avg_order_review_score",
    ]
].describe()

,delivery_deviation_days,avg_order_review_score
count,95824.000000,95824.000000
mean,-11.913070,4.156158
std,10.111737,1.283615
min,-147.000000,1.000000
25%,-17.000000,4.000000
50%,-12.000000,5.000000
75%,-7.000000,5.000000
max,188.000000,5.000000


In [24]:
len(delivery_review)

95824

In [25]:
delivery_review_result = spearman_test(
    data=delivery_review,
    x="delivery_deviation_days",
    y="avg_order_review_score",
)

delivery_review_result

{'n': 95824,
 'rho': np.float64(-0.17617247976426434),
 'p_value': np.float64(0.0)}

In [26]:
test_results["delivery_review"] = {
    "hypothesis":
        "Delivery deviation ↔ review score",

    **delivery_review_result,
}

In [27]:
rho = delivery_review_result["rho"]
p_value = delivery_review_result["p_value"]

print(
    f"n = {delivery_review_result['n']:,}"
)

print(
    f"Spearman rho = {rho:.4f}"
)

if p_value < 0.001:
    print("p-value < 0.001")
else:
    print(f"p-value = {p_value:.4f}")

n = 95,824
Spearman rho = -0.1762
p-value < 0.001


In [28]:
delivery_review_reject = (
    delivery_review_result["p_value"]
    < ALPHA
)

delivery_review_reject

np.True_

In [29]:
if delivery_review_reject:
    print(
        "H₀ отвергается: "
        "между признаками обнаружена "
        "статистически значимая связь."
    )
else:
    print(
        "Нет оснований отвергать H₀."
    )

H₀ отвергается: между признаками обнаружена статистически значимая связь.


### Вывод

На выборке из 95 824 доставленных заказов обнаружена статистически значимая отрицательная монотонная связь между отклонением от ожидаемой даты доставки и оценкой заказа (`Spearman ρ = -0.1762`, `p < 0.001`).

На уровне значимости `α = 0.05` нулевая гипотеза об отсутствии монотонной связи отвергается.

При этом величина коэффициента невелика, поэтому связь следует характеризовать как слабую. Чем позже заказ доставлен относительно обещанной даты, тем в среднем ниже его оценка, однако сама связь между этими признаками невелика.

Окончательное решение о статистической значимости будет принято после применения поправки Холма ко всем четырём проверяемым гипотезам.

## 5. Гипотеза 2: расстояние и стоимость доставки

Проверим статистическую связь между приблизительным расстоянием от продавца до покупателя (`distance_km`) и стоимостью доставки для соответствующей пары `order_id × seller_id` (`pair_freight_value`).

**H₀:** между расстоянием и стоимостью доставки отсутствует монотонная связь.

**H₁:** между расстоянием и стоимостью доставки существует монотонная связь.

На основании результатов EDA ожидается положительное направление связи: большие расстояния соответствуют более высокой стоимости доставки.

Единица анализа — пара `order_id × seller_id` с валидными географическими координатами.

Поскольку один заказ может содержать несколько продавцов, для статистического вывода учитывается кластеризация наблюдений по `order_id`.

In [30]:
pairs_per_order = (
    geo_pairs
    .groupby("order_id")
    .size()
)

pairs_per_order.describe()

count    95439.000000
mean         1.013778
std          0.123043
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          5.000000
dtype: float64

In [31]:
pd.Series({
    "pairs": len(geo_pairs),
    "orders": geo_pairs["order_id"].nunique(),
    "multi_seller_orders": (
        pairs_per_order > 1
    ).sum(),
})

pairs                  96754
orders                 95439
multi_seller_orders     1250
dtype: int64

In [32]:
def clustered_spearman_test(
    data,
    x,
    y,
    cluster,
):
    sample = (
        data[
            [
                x,
                y,
                cluster,
            ]
        ]
        .dropna()
        .copy()
    )

    # Ранги признаков.
    sample["x_rank"] = (
        sample[x]
        .rank(method="average")
    )

    sample["y_rank"] = (
        sample[y]
        .rank(method="average")
    )

    # Стандартизация рангов.
    sample["x_rank_std"] = (
        sample["x_rank"]
        - sample["x_rank"].mean()
    ) / sample["x_rank"].std(ddof=0)

    sample["y_rank_std"] = (
        sample["y_rank"]
        - sample["y_rank"].mean()
    ) / sample["y_rank"].std(ddof=0)

    X = sm.add_constant(
        sample["x_rank_std"]
    )

    model = sm.OLS(
        sample["y_rank_std"],
        X,
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": sample[cluster],
            "use_correction": True,
        },
        use_t=True,
    )

    rho = stats.spearmanr(
        sample[x],
        sample[y],
    ).statistic

    # При стандартизированных рангах
    # коэффициент регрессии равен Spearman rho.
    assert np.isclose(
        model.params["x_rank_std"],
        rho,
    )

    ci = (
        model
        .conf_int(
            alpha=1 - CONFIDENCE_LEVEL
        )
        .loc["x_rank_std"]
    )

    return {
        "n": len(sample),
        "clusters": sample[
            cluster
        ].nunique(),
        "rho": rho,
        "ci_low": ci.iloc[0],
        "ci_high": ci.iloc[1],
        "p_value":
            model.pvalues["x_rank_std"],
    }

In [33]:
distance_freight_result = (
    clustered_spearman_test(
        data=geo_pairs,
        x="distance_km",
        y="pair_freight_value",
        cluster="order_id",
    )
)

distance_freight_result

{'n': 96754,
 'clusters': 95439,
 'rho': np.float64(0.5816623243016442),
 'ci_low': np.float64(0.576789268017772),
 'ci_high': np.float64(0.5865353805855187),
 'p_value': np.float64(0.0)}

In [34]:
test_results["distance_freight"] = {
    "hypothesis":
        "Distance ↔ freight value",

    **distance_freight_result,
}

In [35]:
rho = distance_freight_result["rho"]
p_value = distance_freight_result["p_value"]

print(
    f"n = {distance_freight_result['n']:,}"
)

print(
    "orders = "
    f"{distance_freight_result['clusters']:,}"
)

print(
    f"Spearman rho = {rho:.4f}"
)

print(
    "cluster-robust 95% CI = "
    f"[{distance_freight_result['ci_low']:.4f}, "
    f"{distance_freight_result['ci_high']:.4f}]"
)

if p_value < 0.001:
    print("p-value < 0.001")
else:
    print(f"p-value = {p_value:.4f}")

n = 96,754
orders = 95,439
Spearman rho = 0.5817
cluster-robust 95% CI = [0.5768, 0.5865]
p-value < 0.001


### Вывод

На выборке из 96 754 пар `order_id × seller_id`, относящихся к 95 439 заказам, обнаружена статистически значимая положительная монотонная связь между расстоянием от продавца до покупателя и стоимостью доставки (`Spearman ρ = 0.5817`, cluster-robust `95% CI [0.5768; 0.5865]`, `p < 0.001`).

На уровне значимости `α = 0.05` нулевая гипотеза об отсутствии монотонной связи отвергается.

Связь является заметной по величине: более удалённые пары продавец–покупатель, как правило, связаны с более высокой стоимостью доставки.

При расчёте статистической значимости и доверительного интервала была учтена зависимость наблюдений внутри одного заказа с помощью кластер-робастной оценки стандартной ошибки по `order_id`.

Окончательное решение о статистической значимости будет принято после применения поправки Холма ко всем четырём проверяемым гипотезам.

## 6. Гипотеза 3: расстояние и время доставки

Проверим статистическую связь между расстоянием до наиболее удалённого продавца (`max_seller_distance_km`) и полным временем доставки заказа (`delivery_time_days`).

Для заказов с несколькими продавцами используется максимальное расстояние до продавца. В анализ включаются только заказы, для которых доступны валидные географические координаты всех продавцов.

**H₀:** между расстоянием до наиболее удалённого продавца и временем доставки отсутствует монотонная связь.

**H₁:** между расстоянием до наиболее удалённого продавца и временем доставки существует монотонная связь.

На основании результатов EDA ожидается положительное направление связи: более удалённые заказы обычно имеют более длительное время доставки.

Единица анализа — доставленный заказ.

In [36]:
distance_delivery[
    [
        "max_seller_distance_km",
        "delivery_time_days",
    ]
].describe()

,max_seller_distance_km,delivery_time_days
count,95411.000000,95411.000000
mean,601.563772,12.553631
std,593.043072,9.550002
min,0.000000,0.533414
25%,186.517874,6.759248
50%,435.024528,10.211944
75%,800.286229,15.714369
max,3710.420173,209.628611


In [37]:
len(distance_delivery)

95411

In [38]:
distance_delivery_result = spearman_test(
    data=distance_delivery,
    x="max_seller_distance_km",
    y="delivery_time_days",
)

distance_delivery_result

{'n': 95411, 'rho': np.float64(0.5412639162751879), 'p_value': np.float64(0.0)}

In [39]:
test_results["distance_delivery"] = {
    "hypothesis":
        "Distance ↔ delivery time",

    **distance_delivery_result,
}

In [40]:
rho = distance_delivery_result["rho"]
p_value = distance_delivery_result["p_value"]

print(
    f"n = {distance_delivery_result['n']:,}"
)

print(
    f"Spearman rho = {rho:.4f}"
)

if p_value < 0.001:
    print("p-value < 0.001")
else:
    print(f"p-value = {p_value:.4f}")

n = 95,411
Spearman rho = 0.5413
p-value < 0.001


### Вывод

На выборке из 95 411 доставленных заказов обнаружена статистически значимая положительная монотонная связь между расстоянием до наиболее удалённого продавца и полным временем доставки заказа (`Spearman ρ = 0.5413`, `p < 0.001`).

На уровне значимости `α = 0.05` нулевая гипотеза об отсутствии монотонной связи отвергается.

Связь является заметной по величине: заказы с большим расстоянием до наиболее удалённого продавца, как правило, связаны с более продолжительным временем доставки.

Полученный результат показывает статистическую ассоциацию между расстоянием и временем доставки и не позволяет сам по себе делать вывод о причинном влиянии расстояния.

Окончательное решение о статистической значимости будет принято после применения поправки Холма ко всем четырём проверяемым гипотезам.

## 7. Гипотеза 4: количество installments и стоимость товаров в заказе

Проверим статистическую связь между количеством installments (`installments`) и стоимостью товаров в заказе (`product_gmv`).

В анализ включены только доставленные заказы с одной валидной платежной записью по кредитной карте. Такое ограничение позволяет однозначно сопоставить значение `payment_installments` со стоимостью товаров всего заказа.

**H₀:** между количеством installments и стоимостью товаров в заказе отсутствует монотонная связь.

**H₁:** между количеством installments и стоимостью товаров в заказе существует монотонная связь.

На основании результатов EDA ожидается положительное направление связи: заказы с большей стоимостью товаров чаще связаны с большим количеством installments.

Единица анализа — доставленный заказ с одной валидной платежной записью по кредитной карте.

Для проверки используется двусторонний тест значимости коэффициента ранговой корреляции Спирмена.

In [41]:
installment_orders[
    [
        "installments",
        "product_gmv",
    ]
].describe()

,installments,product_gmv
count,71841.000000,71841.000000
mean,3.540346,142.283913
std,2.849792,210.356499
min,1.000000,0.850000
25%,1.000000,49.000000
50%,3.000000,89.900000
75%,5.000000,156.990000
max,24.000000,13440.000000


In [42]:
len(installment_orders)

71841

In [43]:
(
    installment_orders["installments"]
    .value_counts()
    .sort_index()
)

installments
1     23113
2     11769
3      9950
4      6752
5      4991
6      3722
7      1534
8      4061
9       608
10     5024
11       22
12      124
13       15
14       11
15       70
16        5
17        7
18       26
20       16
21        3
22        1
23        1
24       16
Name: count, dtype: int64

In [44]:
installment_gmv_result = spearman_test(
    data=installment_orders,
    x="installments",
    y="product_gmv",
)

installment_gmv_result

{'n': 71841, 'rho': np.float64(0.4485570792196427), 'p_value': np.float64(0.0)}

In [45]:
test_results["installment_gmv"] = {
    "hypothesis":
        "Installments ↔ product GMV",

    **installment_gmv_result,
}

In [46]:
rho = installment_gmv_result["rho"]
p_value = installment_gmv_result["p_value"]

print(
    f"n = {installment_gmv_result['n']:,}"
)

print(
    f"Spearman rho = {rho:.4f}"
)

if p_value < 0.001:
    print("p-value < 0.001")
else:
    print(f"p-value = {p_value:.4f}")

n = 71,841
Spearman rho = 0.4486
p-value < 0.001


### Вывод

На выборке из 71 841 доставленного заказа с одной валидной платежной записью по кредитной карте обнаружена статистически значимая положительная монотонная связь между количеством installments и стоимостью товаров в заказе (`Spearman ρ = 0.4486`, `p < 0.001`).

На уровне значимости `α = 0.05` нулевая гипотеза об отсутствии монотонной связи отвергается.

Связь является умеренной по величине: заказы с большей стоимостью товаров, как правило, связаны с большим количеством installments.

Полученный результат отражает статистическую ассоциацию и не позволяет сделать вывод о том, что увеличение стоимости заказа само по себе приводит к выбору большего количества installments или наоборот.

Результат относится только к выбранной подвыборке доставленных заказов с одной валидной платежной записью по кредитной карте и не обязательно распространяется на заказы с несколькими платежными записями или смешанными способами оплаты.

Окончательное решение о статистической значимости будет принято после применения поправки Холма ко всем четырём проверяемым гипотезам.

## 8. Поправка на множественные проверки

В рамках анализа одновременно проверяются четыре статистические гипотезы.

При выполнении нескольких статистических тестов вероятность хотя бы одного ложноположительного результата возрастает. Поэтому для контроля семейной вероятности ошибки первого рода (`FWER`) применяется поправка Холма.

Метод Холма последовательно корректирует `p-value` и является менее консервативным, чем простая поправка Бонферрони, сохраняя контроль `FWER` на уровне `α = 0.05`.

Коррекция применяется ко всем четырём гипотезам как к одной семье статистических проверок.

Важно, что поправка Холма учитывает множественность проведённых тестов, но не устраняет ограничение, связанное с тем, что сами гипотезы были сформулированы после EDA на тех же данных.

In [47]:
results_df = (
    pd.DataFrame
    .from_dict(
        test_results,
        orient="index",
    )
    .reset_index()
    .rename(
        columns={
            "index": "test_id",
        }
    )
)

results_df[
    [
        "test_id",
        "hypothesis",
        "n",
        "rho",
        "p_value",
    ]
]

,test_id,hypothesis,n,rho,p_value
0,delivery_review,Delivery deviation ↔ review score,95824,-0.176172,0.0
1,distance_freight,Distance ↔ freight value,96754,0.581662,0.0
2,distance_delivery,Distance ↔ delivery time,95411,0.541264,0.0
3,installment_gmv,Installments ↔ product GMV,71841,0.448557,0.0


In [48]:
(
    reject_holm,
    p_value_holm,
    _,
    _,
) = multipletests(
    results_df["p_value"],
    alpha=ALPHA,
    method="holm",
)

In [49]:
results_df["p_value_holm"] = (
    p_value_holm
)

results_df["reject_h0_holm"] = (
    reject_holm
)

In [50]:
results_df[
    [
        "hypothesis",
        "rho",
        "p_value",
        "p_value_holm",
        "reject_h0_holm",
    ]
]

,hypothesis,rho,p_value,p_value_holm,reject_h0_holm
0,Delivery deviation ↔ review score,-0.176172,0.0,0.0,True
1,Distance ↔ freight value,0.581662,0.0,0.0,True
2,Distance ↔ delivery time,0.541264,0.0,0.0,True
3,Installments ↔ product GMV,0.448557,0.0,0.0,True


In [51]:
def format_p_value(p_value):
    if p_value < 0.001:
        return "< 0.001"

    return f"{p_value:.4f}"

In [52]:
results_summary = (
    results_df[
        [
            "hypothesis",
            "n",
            "rho",
            "p_value",
            "p_value_holm",
            "reject_h0_holm",
        ]
    ]
    .copy()
)

results_summary[
    "p_value"
] = (
    results_summary[
        "p_value"
    ]
    .map(format_p_value)
)

results_summary[
    "p_value_holm"
] = (
    results_summary[
        "p_value_holm"
    ]
    .map(format_p_value)
)

results_summary

,hypothesis,n,rho,p_value,p_value_holm,reject_h0_holm
0,Delivery deviation ↔ review score,95824,-0.176172,< 0.001,< 0.001,True
1,Distance ↔ freight value,96754,0.581662,< 0.001,< 0.001,True
2,Distance ↔ delivery time,95411,0.541264,< 0.001,< 0.001,True
3,Installments ↔ product GMV,71841,0.448557,< 0.001,< 0.001,True


### Вывод

После применения поправки Холма все четыре скорректированных `p-value` остаются ниже уровня значимости `α = 0.05`.

Следовательно, нулевые гипотезы об отсутствии монотонной связи отвергаются для всех четырёх проверяемых взаимосвязей.

Поправка на множественные проверки не изменила статистические решения, полученные на основе исходных `p-value`.

При этом статистическая значимость не означает одинаковую силу обнаруженных связей. Наиболее выраженные связи наблюдаются между расстоянием и стоимостью доставки (`ρ ≈ 0.58`) и между расстоянием и временем доставки (`ρ ≈ 0.54`). Связь количества installments со стоимостью товаров умеренная (`ρ ≈ 0.45`), а связь своевременности доставки с оценкой заказа значительно слабее (`ρ ≈ -0.18`).

Практическая значимость этих зависимостей требует отдельной интерпретации в бизнес-контексте.

## 9. Итоговые выводы и ограничения

Статистический анализ выявил статистически значимые монотонные связи для всех четырёх зависимостей, сформулированных по результатам EDA. После применения поправки Холма все результаты остаются статистически значимыми при уровне `α = 0.05`.

Основные результаты:

- между отклонением фактической даты доставки от ожидаемой и оценкой заказа наблюдается слабая отрицательная связь (`ρ ≈ -0.18`);
- между расстоянием от продавца до покупателя и стоимостью доставки наблюдается положительная связь (`ρ ≈ 0.58`);
- между расстоянием до наиболее удалённого продавца и полным временем доставки наблюдается положительная связь (`ρ ≈ 0.54`);
- между количеством installments и стоимостью товаров в заказе наблюдается умеренная положительная связь (`ρ ≈ 0.45`).

Наиболее выраженные зависимости связаны с географическим расстоянием: более удалённые пары продавец–покупатель связаны с более высокой стоимостью доставки, а заказы с большим расстоянием до наиболее удалённого продавца — с большей продолжительностью доставки.

Связь между отклонением от обещанного срока доставки и оценкой заказа статистически значима, однако невелика по величине. Это показывает, что своевременность доставки связана с оценкой заказа, но сама по себе эта связь выражена слабо.

### Ограничения анализа

- Гипотезы были сформулированы после EDA на тех же данных, поэтому результаты следует рассматривать как исследовательские, а не как независимое подтверждение заранее зарегистрированных гипотез.
- Наблюдаемые связи являются ассоциациями и сами по себе не позволяют делать причинно-следственные выводы.
- Географическое расстояние является приблизительным и рассчитывается по агрегированным координатам ZIP-префиксов, а не по фактическому транспортному маршруту.
- Для анализа расстояния и стоимости доставки зависимость нескольких продавцов внутри одного заказа учитывалась с помощью кластер-робастной оценки по `order_id`. Возможная зависимость между заказами одного продавца отдельно не моделировалась.
- В order-level анализах отдельные заказы рассматриваются как независимые наблюдения, хотя один покупатель может совершить несколько заказов.
- Анализ installments относится только к доставленным заказам с одной валидной платежной записью по кредитной карте и не обязательно распространяется на заказы со смешанными способами оплаты или несколькими платежными записями.
- Большой размер выборок делает возможной статистическую значимость даже сравнительно слабых связей, поэтому результаты интерпретируются совместно с величиной коэффициента корреляции.